# Postgres to Postgres with logical replication

Change data capture without Debezium, Kafka or any tool outside Postgres. The source publishes
tables, the replica subscribes, and every committed insert, update and delete arrives on the
replica a few milliseconds later. This is the mechanism Debezium, AWS DMS and Redshift zero-ETL
all sit on; they read the same stream and write it somewhere other than Postgres.

Two servers from `docker/postgres-pgvector.yaml`: `pgvector-dev` (the source, port 5432, running
with `wal_level=logical`) and `pgvector-replica` (port 5440, `--profile replica`). Both come from
the same image, so pgvector is on both and `vector` columns replicate.

    just db replica          # starts both
    just nb logical_replication

Roles: the source needs a role with `REPLICATION` (`app` is a superuser here, which covers it).
On RDS that is `rds_replication`, and `rds.logical_replication = 1` in the parameter group replaces
`wal_level=logical`.

In [1]:
# Parameters (papermill -p): which two servers from src/settings.py play source and replica.
SOURCE = "pg16"
REPLICA = "replica"
N_BURST = 20_000  # rows written while the subscription is paused, to watch WAL pile up

In [2]:
import pathlib
import subprocess
import sys
import time

import pandas as pd
import psycopg
from dotenv import load_dotenv

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "db.py").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

from settings import settings  # noqa: E402

def dsn(server: str) -> str:
    return settings.sqlalchemy_url(server).replace("postgresql+psycopg://", "postgresql://")

SRC, REP = dsn(SOURCE), dsn(REPLICA)

# start both containers if needed (the replica only exists with its compose profile)
subprocess.run(["docker", "compose", "-f", "docker/postgres-pgvector.yaml", "--profile", "replica", "up", "-d"], cwd=ROOT, check=True, capture_output=True)
for name, d in (("source", SRC), ("replica", REP)):
    for _ in range(60):
        try:
            with psycopg.connect(d, connect_timeout=2) as c:
                print(f"{name:<8} {d.split('@')[-1]:<20} {c.execute('SELECT version()').fetchone()[0][:18]}  wal_level={c.execute('SHOW wal_level').fetchone()[0]}")
            break
        except psycopg.OperationalError:
            time.sleep(1)
    else:
        raise RuntimeError(f"{name} at {d} does not answer")

def q(d: str, sql: str, *args) -> pd.DataFrame:
    """One query as a DataFrame."""
    with psycopg.connect(d) as c:
        cur = c.execute(sql, args or None)
        return pd.DataFrame(cur.fetchall(), columns=[col.name for col in cur.description])

def run(d: str, sql: str, *args, autocommit: bool = False) -> None:
    """One statement; autocommit=True for the ones Postgres refuses inside a transaction."""
    with psycopg.connect(d, autocommit=autocommit) as c:
        c.execute(sql, args or None)

source   localhost:5432/app   PostgreSQL 16.15 (  wal_level=logical
replica  localhost:5440/app   PostgreSQL 16.15 (  wal_level=replica


## 1. Publish on the source

A publication is a named list of tables (and, since Postgres 15, optional row filters and column
lists). It changes nothing by itself: it is the menu a subscriber orders from. The tables need a
primary key or a `REPLICA IDENTITY`, otherwise updates and deletes cannot be replicated because
the replica would not know which row to change.

The tables come from the project's models, so `events` (JSONB) and `documents` (a 1536-dim
`vector`) exist on the source; a fresh `cdc_demo` table is used for the counting so nothing in the
playground tables is touched.

In [3]:
from sqlalchemy import create_engine

from models import Base  # noqa: E402  (Document, Event)

create_engine(settings.sqlalchemy_url(SOURCE)).dispose()  # just to fail early if the URL is wrong
with psycopg.connect(SRC, autocommit=True) as c:
    c.execute("CREATE EXTENSION IF NOT EXISTS vector")
Base.metadata.create_all(create_engine(settings.sqlalchemy_url(SOURCE)))

run(SRC, "DROP TABLE IF EXISTS cdc_demo", autocommit=True)
run(SRC, "CREATE TABLE cdc_demo (id bigserial PRIMARY KEY, kind text NOT NULL, n int NOT NULL, note text)", autocommit=True)
run(SRC, "DROP PUBLICATION IF EXISTS app_pub", autocommit=True)
run(SRC, "CREATE PUBLICATION app_pub FOR TABLE cdc_demo, events, documents", autocommit=True)
q(SRC, "SELECT pubname, schemaname, tablename, attnames, rowfilter FROM pg_publication_tables WHERE pubname = 'app_pub'")

,pubname,schemaname,tablename,attnames,rowfilter
0,app_pub,public,documents,"[id, content, source, embedding, created_at]",None
1,app_pub,public,events,"[id, kind, payload, created_at]",None
2,app_pub,public,cdc_demo,"[id, kind, n, note]",None


## 2. Subscribe on the replica

DDL is not replicated, ever. The replica must already have tables with the same names and at
least the published columns, and the same extensions for the column types. Here the models
create them, the same way a migration tool would in production.

`CREATE SUBSCRIPTION` connects to the source, creates a replication slot there (named after the
subscription), copies the existing rows of each table, then starts applying changes from the slot.
It refuses to run inside a transaction, hence `autocommit=True`. The connection string points at
the source as the replica sees it: on the compose network that host is `db`, not `localhost`.

The target tables must be empty, or hold nothing the copy will also bring: the initial copy is a
`COPY` into the table, it does not merge, and a duplicate key makes the sync worker fail and retry
until someone clears the row.

In [4]:
with psycopg.connect(REP, autocommit=True) as c:
    c.execute("CREATE EXTENSION IF NOT EXISTS vector")
    c.execute("DROP SUBSCRIPTION IF EXISTS app_sub")  # also drops its slot on the source
    c.execute("DROP TABLE IF EXISTS cdc_demo")
    c.execute("CREATE TABLE cdc_demo (id bigserial PRIMARY KEY, kind text NOT NULL, n int NOT NULL, note text)")
Base.metadata.create_all(create_engine(settings.sqlalchemy_url(REPLICA)))
# the initial copy appends: rows already on the replica collide with it and the sync worker retries forever
run(REP, "TRUNCATE cdc_demo, events, documents", autocommit=True)

t0 = time.perf_counter()
run(REP, """
    CREATE SUBSCRIPTION app_sub
    CONNECTION 'host=db port=5432 dbname=app user=app password=pwd'
    PUBLICATION app_pub
""", autocommit=True)

# the initial copy: every table goes i (init) -> d (data copy) -> s/f (sync) -> r (ready)
while True:
    st = q(REP, "SELECT srrelid::regclass AS tbl, srsubstate AS state FROM pg_subscription_rel")
    if set(st.state) == {"r"}:
        break
    time.sleep(0.2)
print(f"subscribed and in sync after {time.perf_counter() - t0:.1f}s")
st

subscribed and in sync after 0.2s


,tbl,state
0,documents,r
1,cdc_demo,r
2,events,r


## 3. Changes flow

Insert, update and delete on the source; count on the replica until it agrees, and time it. The
replica sees whole transactions in commit order, never a half-applied one. A `vector` column comes
across unchanged because both sides know the type.

In [5]:
def wait_for(pred, timeout: float = 10.0) -> float:
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < timeout:
        if pred():
            return (time.perf_counter() - t0) * 1000
        time.sleep(0.005)
    raise TimeoutError("replica did not catch up")

with psycopg.connect(SRC) as c:
    c.execute("INSERT INTO cdc_demo (kind, n, note) SELECT 'insert', g, NULL FROM generate_series(1, 1000) g")
    c.commit()
ms_insert = wait_for(lambda: q(REP, "SELECT count(*) AS n FROM cdc_demo").n[0] == 1000)

with psycopg.connect(SRC) as c:
    c.execute("UPDATE cdc_demo SET note = 'touched', kind = 'update' WHERE n % 10 = 0")
    c.commit()
ms_update = wait_for(lambda: q(REP, "SELECT count(*) AS n FROM cdc_demo WHERE kind = 'update'").n[0] == 100)

with psycopg.connect(SRC) as c:
    c.execute("DELETE FROM cdc_demo WHERE n > 900")
    c.commit()
ms_delete = wait_for(lambda: q(REP, "SELECT count(*) AS n FROM cdc_demo").n[0] == 900)

# a document with an embedding, read back on the replica through pgvector
import numpy as np  # noqa: E402
from pgvector.psycopg import register_vector  # noqa: E402

v = np.random.default_rng(1).random(1536, dtype=np.float32)
with psycopg.connect(SRC) as c:
    register_vector(c)
    c.execute("DELETE FROM documents WHERE source = 'cdc'")
    c.execute("INSERT INTO documents (id, content, source, embedding) VALUES (gen_random_uuid(), 'replicated row', 'cdc', %s)", (v,))
    c.commit()
def _same() -> bool:
    with psycopg.connect(REP) as c:
        register_vector(c)
        row = c.execute("SELECT embedding FROM documents WHERE source = 'cdc'").fetchone()
        return row is not None and np.allclose(row[0].to_numpy(), v)
ms_vector = wait_for(_same)

pd.DataFrame({"change": ["1000 inserts", "100 updates", "100 deletes", "1 vector(1536) row"],
              "visible on replica after (ms)": [round(x, 1) for x in (ms_insert, ms_update, ms_delete, ms_vector)]})

,change,visible on replica after (ms)
0,1000 inserts,7.2
1,100 updates,6.1
2,100 deletes,5.7
3,1 vector(1536) row,16.3


## 4. What to monitor

Two views, one per side. On the source, `pg_replication_slots` shows every slot and how much WAL
it is holding back: `confirmed_flush_lsn` is what the subscriber has acknowledged, and everything
after it stays on disk until it does. On the replica, `pg_stat_subscription` shows the apply worker
and how far behind it is.

In [6]:
print("source: slots")
display(q(SRC, """
    SELECT slot_name, plugin, active, wal_status,
           pg_size_pretty(pg_wal_lsn_diff(pg_current_wal_lsn(), confirmed_flush_lsn)) AS wal_retained
    FROM pg_replication_slots
"""))
print("replica: subscription workers")
q(REP, """
    SELECT subname, pid IS NOT NULL AS worker_running,
           pg_size_pretty(pg_wal_lsn_diff(latest_end_lsn, received_lsn)) AS behind,
           now() - latest_end_time AS since_last_apply
    FROM pg_stat_subscription
""")

source: slots


,slot_name,plugin,active,wal_status,wal_retained
0,app_sub,pgoutput,True,reserved,183 kB


replica: subscription workers


,subname,worker_running,behind,since_last_apply
0,app_sub,True,0 bytes,0 days 00:00:00.048820


## 5. The slot holds WAL: pause the subscriber and watch

This is the failure that takes production down. A slot that nobody reads from never advances, and
Postgres cannot recycle WAL past it, so the disk fills. It happens when the subscriber, Debezium or
DMS task is stopped and forgotten. The demo disables the subscription, writes a burst, shows the
retained WAL growing on the source, then re-enables and times the catch-up.

`max_slot_wal_keep_size` (Postgres 13+) caps this: past the cap the slot is invalidated and the
subscriber has to start over with a fresh copy. Losing a slot beats losing the primary.

In [7]:
run(REP, "ALTER SUBSCRIPTION app_sub DISABLE", autocommit=True)
time.sleep(1)
before = q(SRC, "SELECT pg_wal_lsn_diff(pg_current_wal_lsn(), confirmed_flush_lsn)::bigint AS b FROM pg_replication_slots WHERE slot_name = 'app_sub'").b[0]

with psycopg.connect(SRC) as c:
    c.execute("INSERT INTO cdc_demo (kind, n, note) SELECT 'burst', g, repeat('x', 200) FROM generate_series(1, %s) g", (N_BURST,))
    c.commit()
after = q(SRC, "SELECT pg_wal_lsn_diff(pg_current_wal_lsn(), confirmed_flush_lsn)::bigint AS b FROM pg_replication_slots WHERE slot_name = 'app_sub'").b[0]
print(f"WAL retained by the slot: {before / 1e6:.1f} MB -> {after / 1e6:.1f} MB after {N_BURST:,} rows nobody consumed")

run(REP, "ALTER SUBSCRIPTION app_sub ENABLE", autocommit=True)
ms = wait_for(lambda: q(REP, "SELECT count(*) AS n FROM cdc_demo WHERE kind = 'burst'").n[0] == N_BURST, timeout=120)
retained = q(SRC, "SELECT pg_wal_lsn_diff(pg_current_wal_lsn(), confirmed_flush_lsn)::bigint AS b FROM pg_replication_slots WHERE slot_name = 'app_sub'").b[0]
print(f"re-enabled: caught up in {ms / 1000:.1f}s, slot now retains {retained / 1e6:.1f} MB")

WAL retained by the slot: 0.0 MB -> 7.0 MB after 20,000 rows nobody consumed


re-enabled: caught up in 0.2s, slot now retains 7.0 MB


## 6. DDL is not replicated: add a column and see it break

Add a column on the source only, then write a row. The apply worker on the replica cannot fit the
row into a table that lacks the column, logs an error, and retries forever; nothing after that row
is applied either, so the slot starts holding WAL again. The fix is the schema change on the
replica; the worker picks up on the next retry. Order matters: in production you change the
subscriber first, then the publisher, because a replica with an extra column is fine and a replica
with a missing one is not.

In [8]:
run(SRC, "ALTER TABLE cdc_demo ADD COLUMN extra int", autocommit=True)
run(SRC, "INSERT INTO cdc_demo (kind, n, extra) VALUES ('ddl', 1, 42)")
time.sleep(3)
errors = q(REP, "SELECT subname, apply_error_count, sync_error_count FROM pg_stat_subscription_stats")
display(errors)
print("replica has the 'ddl' row:", q(REP, "SELECT count(*) AS n FROM cdc_demo WHERE kind = 'ddl'").n[0] == 1)

run(REP, "ALTER TABLE cdc_demo ADD COLUMN extra int", autocommit=True)
ms = wait_for(lambda: q(REP, "SELECT count(*) AS n FROM cdc_demo WHERE kind = 'ddl' AND extra = 42").n[0] == 1, timeout=30)
print(f"after adding the column on the replica: applied within {ms / 1000:.1f}s")

,subname,apply_error_count,sync_error_count
0,app_sub,1,0


replica has the 'ddl' row: False


after adding the column on the replica: applied within 1.7s


## 7. Conflicts: the replica is not read-only

Nothing stops a client writing to the replica's tables. When the source later inserts the same
primary key, the apply worker hits a unique violation and stops, exactly like the missing column.
The cure is the same, remove the offending row on the replica, or tell the subscription to skip
that transaction with `ALTER SUBSCRIPTION ... SKIP (lsn = ...)` using the LSN from the replica's
log. Postgres 16 logs conflicts with that LSN; Postgres 18 adds `pg_stat_subscription_stats`
counters per conflict type. The real fix is not to write to replicated tables on the replica.

In [9]:
run(REP, "INSERT INTO cdc_demo (id, kind, n) VALUES (999999, 'local write', 0)")
run(SRC, "INSERT INTO cdc_demo (id, kind, n) VALUES (999999, 'from source', 0)")
time.sleep(3)
print("apply errors:", q(REP, "SELECT apply_error_count FROM pg_stat_subscription_stats").apply_error_count[0])

run(REP, "DELETE FROM cdc_demo WHERE id = 999999")  # resolve: the source wins
ms = wait_for(lambda: q(REP, "SELECT kind FROM cdc_demo WHERE id = 999999").kind.tolist() == ["from source"], timeout=30)
print(f"resolved: source row applied {ms / 1000:.1f}s after the conflicting row was removed")

apply errors: 2


resolved: source row applied 1.9s after the conflicting row was removed


## 8. What else is not replicated, and the tools built on this

- **Sequences.** The replica's `bigserial` counters never move. After a failover to the replica,
  reset them: `SELECT setval('cdc_demo_id_seq', (SELECT max(id) FROM cdc_demo))`.
- **DDL, TRUNCATE by default in older versions, large objects.** Schema changes go through your
  migration tool on both sides, subscriber first.
- **Failover of the source.** Before Postgres 17 the slot lives only on the primary; after a
  failover the subscription has to be recreated. Postgres 17 has failover slots.
- **Row and column filters** (Postgres 15+): `CREATE PUBLICATION p FOR TABLE events (id, kind)
  WHERE (kind = 'purchase')`. The filter only sees the row's own columns, and the replica identity
  columns must be included.

The same stream, consumed by something other than Postgres:

| tool | reads the slot with | writes to |
|---|---|---|
| Debezium | pgoutput, as a Kafka Connect source | Kafka topics, one per table; sinks from there |
| AWS DMS | pgoutput or test_decoding | Redshift, S3, Aurora, DynamoDB, Kafka, ... |
| Redshift zero-ETL | pgoutput, managed | Redshift, from RDS or Aurora PostgreSQL only |
| pg_recvlogical | any plugin, a CLI | stdout or a file; good for looking at the raw stream |
| Apache Doris 4.1 streaming job | pgoutput, Debezium's connector inside the BE | Doris tables it creates itself; see `doris_cdc.ipynb` |

On RDS the source and the subscriber can both be RDS instances: set `rds.logical_replication = 1`
in the parameter group (a reboot), grant `rds_replication` to the role, and open the security
group between them. `rds_superuser` can create publications and subscriptions.

## Cleanup

Dropping the subscription drops its slot on the source, which is the part that matters: a
forgotten slot keeps WAL forever. The containers stay up; `just db-down replica -v` removes both
and their data.

In [10]:
run(REP, "DROP SUBSCRIPTION IF EXISTS app_sub", autocommit=True)
run(SRC, "DROP PUBLICATION IF EXISTS app_pub", autocommit=True)
run(SRC, "DROP TABLE IF EXISTS cdc_demo", autocommit=True)
run(REP, "DROP TABLE IF EXISTS cdc_demo", autocommit=True)
run(SRC, "DELETE FROM documents WHERE source = 'cdc'")
print("slots left on the source:", q(SRC, "SELECT slot_name FROM pg_replication_slots").slot_name.tolist())

slots left on the source: []
